In [ ]:
# ==========================================================
# Yield Model (CPU, RF + XGB)
# Spatial CV for tuning + Wave-based Train/Test Split
# NeurIPS-ready: clean train/test + seaborn plots
# ==========================================================

import os
import numpy as np
import pandas as pd
import joblib

from tqdm import tqdm
from time import perf_counter

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, ParameterGrid
from sklearn.metrics import r2_score, mean_squared_error

from xgboost import XGBRegressor

import matplotlib.pyplot as plt
import seaborn as sns

# ----------------------------------------------------------
# 0. Paths and output directory
# ----------------------------------------------------------
SAT_PATH   = "/Users/jjburrell/Downloads/EE_harvest_ml.csv"  # your feature file
YIELD_PATH = "/Users/jjburrell/Downloads/Plot_dataset.dta"
MODEL_DIR  = "./models_spatialcv_waveholdout_cpu"
os.makedirs(MODEL_DIR, exist_ok=True)

sns.set_theme(style="whitegrid")

# ----------------------------------------------------------
# 1. Load and preprocess data
# ----------------------------------------------------------
print("🔹 Loading datasets...")
sat_df   = pd.read_csv(SAT_PATH)
yield_df = pd.read_stata(YIELD_PATH)

# Filter out implausible yields and missing coords
yield_df = yield_df[(yield_df["yield_kg"] > 0) & (yield_df["yield_kg"] < 15000)]
yield_df = yield_df.dropna(subset=["yield_kg", "lat_modified", "lon_modified"])

# Aggregate to (country, wave, season, lat, lon)
agg_yields = (
    yield_df.groupby(["country", "wave", "season", "lat_modified", "lon_modified"])
    .agg(mean_yield=("yield_kg", "mean"))
    .reset_index()
)

# Log-transform target
agg_yields["log_yield"] = np.log1p(agg_yields["mean_yield"])

# Merge with satellite data
df = pd.merge(
    agg_yields,
    sat_df,
    on=["country", "wave", "season", "lat_modified", "lon_modified"],
    how="inner",
)

df = df.dropna()
print(f"✅ Final merged shape: {df.shape}")

# ----------------------------------------------------------
# 2. Define variables and outer train/test split
#    Outer split = hold out last wave as test
# ----------------------------------------------------------
merge_vars = ["country", "wave", "season", "lat_modified", "lon_modified"]
y_var = "log_yield"
x_vars = [c for c in df.columns if c not in merge_vars + ["mean_yield", "log_yield"]]

X_full = df[x_vars]
y_full = df[y_var]

# Spatial grouping for inner CV
groups_full = (
    df["country"].astype(str)
    + "_"
    + df["wave"].astype(str)
    + "_"
    + df["lat_modified"].astype(str)
    + "_"
    + df["lon_modified"].astype(str)
)

cv = GroupKFold(n_splits=5)
countries = df["country"].unique()

# Wave-based holdout
test_wave = df["wave"].max()
print(f"📌 Using wave={test_wave} as held-out TEST set.")

train_mask = df["wave"] != test_wave
test_mask  = df["wave"] == test_wave

X_train, y_train, groups_train = (
    X_full.loc[train_mask],
    y_full.loc[train_mask],
    groups_full.loc[train_mask],
)
X_test, y_test, groups_test = (
    X_full.loc[test_mask],
    y_full.loc[test_mask],
    groups_full.loc[test_mask],
)

print("Train size:", X_train.shape, "Test size:", X_test.shape)
print("Countries:", countries)

# ----------------------------------------------------------
# 3. Parameter grids
# ----------------------------------------------------------
rf_param_grid = {
    "n_estimators": [300, 600],
    "max_depth": [12, 24],
    "max_features": [0.5, 0.8],
    "min_samples_split": [2, 8],
    "min_samples_leaf": [1, 4],
}

xgb_param_grid = {
    "n_estimators": [300, 600],
    "max_depth": [4, 6],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "reg_lambda": [1.0, 5.0],
}

def shuffle_grid(param_grid, seed=42):
    grid_list = list(ParameterGrid(param_grid))
    rng = np.random.default_rng(seed)
    return [grid_list[i] for i in rng.permutation(len(grid_list))]

rf_grid_list  = shuffle_grid(rf_param_grid, seed=42)
xgb_grid_list = shuffle_grid(xgb_param_grid, seed=1337)

print(f"Total RF combos:  {len(rf_grid_list)}")
print(f"Total XGB combos: {len(xgb_grid_list)}")

# ----------------------------------------------------------
# 4. Model builders
# ----------------------------------------------------------
def build_model(model_type, params):
    if model_type == "RF":
        return RandomForestRegressor(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            max_features=params["max_features"],
            min_samples_split=params["min_samples_split"],
            min_samples_leaf=params["min_samples_leaf"],
            n_jobs=-1,
            random_state=42,
        )
    elif model_type == "XGB":
        return XGBRegressor(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            learning_rate=params["learning_rate"],
            subsample=params["subsample"],
            colsample_bytree=params["colsample_bytree"],
            reg_lambda=params["reg_lambda"],
            objective="reg:squarederror",
            tree_method="hist",
            n_jobs=-1,
            random_state=42,
        )
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

# ----------------------------------------------------------
# 5. Spatial CV tuning (inner loop, on TRAIN only)
# ----------------------------------------------------------
def tune_with_spatial_cv(
    X, y, groups, grid_list,
    model_type,
    patience=15,
    min_delta=1e-3,
    desc_label="Tuning"
):
    best_r2 = -np.inf
    best_params = None
    no_improve = 0
    start = perf_counter()

    X_values = X.values
    y_values = y.values

    for p in tqdm(grid_list, desc=f"{desc_label} ({model_type})", ncols=100):
        r2_fold = []
        rmse_fold = []

        for tr_idx, va_idx in cv.split(X_values, y_values, groups):
            model = build_model(model_type, p)
            model.fit(X_values[tr_idx], y_values[tr_idx])
            pred = model.predict(X_values[va_idx])
            r2_fold.append(r2_score(y_values[va_idx], pred))
            rmse_fold.append(np.sqrt(mean_squared_error(y_values[va_idx], pred)))

        r2_mean = float(np.mean(r2_fold))

        if r2_mean > best_r2 + min_delta:
            best_r2 = r2_mean
            best_params = p
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"⏹️ Early stopping after {no_improve} non-improving combos.")
                break

    elapsed = perf_counter() - start
    print(f"✅ [{model_type}] Best CV R²={best_r2:.3f} | time={elapsed:.1f}s")
    return best_params, best_r2

def fit_best_model(X, y, model_type, best_params):
    model = build_model(model_type, best_params)
    model.fit(X.values, y.values)
    return model

# ----------------------------------------------------------
# 6. GLOBAL models: tune on TRAIN, evaluate on held-out TEST
# ----------------------------------------------------------
global_models = {}
global_cv_scores = {}

for model_type, grid_list in [("RF", rf_grid_list), ("XGB", xgb_grid_list)]:
    print(f"\n🔧 Tuning GLOBAL {model_type} on TRAIN (spatial CV)...")
    best_params, best_cv_r2 = tune_with_spatial_cv(
        X_train, y_train, groups_train, grid_list,
        model_type=model_type,
        patience=10,
        min_delta=1e-3,
        desc_label="GLOBAL"
    )
    print(f"Best GLOBAL {model_type} params:", best_params)
    global_cv_scores[model_type] = best_cv_r2

    model = fit_best_model(X_train, y_train, model_type, best_params)
    global_models[model_type] = model
    joblib.dump(model, os.path.join(MODEL_DIR, f"model_GLOBAL_{model_type}.joblib"))

# Evaluate global models on held-out test wave per country
results = []

for model_type, model in global_models.items():
    for c in countries:
        mask_c_test = (df["country"] == c) & test_mask
        Xc_test = X_full.loc[mask_c_test]
        yc_test = y_full.loc[mask_c_test]

        if len(Xc_test) == 0:
            continue

        y_pred = model.predict(Xc_test.values)
        r2 = r2_score(yc_test.values, y_pred)
        rmse = np.sqrt(mean_squared_error(yc_test.values, y_pred))

        results.append({
            "country": c,
            "algo": model_type,
            "train_scope": "GLOBAL",
            "r2": r2,
            "rmse": rmse,
            "split": "test",
        })

# ----------------------------------------------------------
# 7. Per-country models: tune on that country's TRAIN subset
#    and evaluate on that country's TEST subset
# ----------------------------------------------------------
for c in countries:
    mask_c_train = (df["country"] == c) & train_mask
    mask_c_test  = (df["country"] == c) & test_mask

    Xc_train = X_full.loc[mask_c_train]
    yc_train = y_full.loc[mask_c_train]
    groups_c_train = groups_full.loc[mask_c_train]

    Xc_test = X_full.loc[mask_c_test]
    yc_test = y_full.loc[mask_c_test]

    if len(Xc_train) < 50 or len(Xc_test) < 10:
        print(f"⚠️ Skipping country-specific models for {c}: "
              f"train={len(Xc_train)}, test={len(Xc_test)}")
        continue

    print(f"\n🔧 Tuning COUNTRY-SPECIFIC models for {c}...")

    for model_type, grid_list in [("RF", rf_grid_list), ("XGB", xgb_grid_list)]:
        print(f"   → {c} {model_type}")
        best_params_c, best_cv_r2_c = tune_with_spatial_cv(
            Xc_train, yc_train, groups_c_train, grid_list,
            model_type=model_type,
            patience=8,
            min_delta=1e-3,
            desc_label=c,
        )
        print(f"   Best {c} {model_type} params:", best_params_c)
        print(f"   {c} {model_type} inner-CV R² (train domain): {best_cv_r2_c:.3f}")

        model_c = fit_best_model(Xc_train, yc_train, model_type, best_params_c)
        joblib.dump(model_c, os.path.join(MODEL_DIR, f"model_{c}_{model_type}.joblib"))

        # Evaluate on held-out test wave for that country
        y_pred_c = model_c.predict(Xc_test.values)
        r2_c = r2_score(yc_test.values, y_pred_c)
        rmse_c = np.sqrt(mean_squared_error(yc_test.values, y_pred_c))

        results.append({
            "country": c,
            "algo": model_type,
            "train_scope": "COUNTRY",
            "r2": r2_c,
            "rmse": rmse_c,
            "split": "test",
        })

# ----------------------------------------------------------
# 8. Save results
# ----------------------------------------------------------
results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(MODEL_DIR, "waveholdout_test_results_cpu.csv"), index=False)

print("\n✅ Saved held-out test results → waveholdout_test_results_cpu.csv")
print(results_df.head())

# ----------------------------------------------------------
# 9. Seaborn plots: Global vs Country, RF and XGB separately
# ----------------------------------------------------------
plot_df = results_df.copy()

# RF plot
rf_df = plot_df[plot_df["algo"] == "RF"]

plt.figure(figsize=(10, 5))
sns.barplot(
    data=rf_df,
    x="country",
    y="r2",
    hue="train_scope",
    palette="Set2",
)
plt.title(f"Held-out Wave {test_wave}: RF – Global vs Country-specific")
plt.ylabel("R² (test)")
plt.xlabel("Country")
plt.xticks(rotation=45)
plt.legend(title="Train Scope", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, "test_r2_by_country_RF.png"), dpi=300)
plt.show()

# XGB plot
xgb_df = plot_df[plot_df["algo"] == "XGB"]

plt.figure(figsize=(10, 5))
sns.barplot(
    data=xgb_df,
    x="country",
    y="r2",
    hue="train_scope",
    palette="Set2",
)
plt.title(f"Held-out Wave {test_wave}: XGB – Global vs Country-specific")
plt.ylabel("R² (test)")
plt.xlabel("Country")
plt.xticks(rotation=45)
plt.legend(title="Train Scope", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, "test_r2_by_country_XGB.png"), dpi=300)
plt.show()

print(f"\nAll models & results saved in: {MODEL_DIR}")
